# 🖼️ Notebook 2 — Image Preprocessing & ViT Feature Extraction

**Breast Ultrasound Report Generation | MSc AI — University of East London**

This notebook covers:
1. Building the TensorFlow image pipeline (resize, normalise, tf.data)
2. Extracting features using a pre-trained Vision Transformer (ViT)
3. Saving features as a `.npy` file for downstream retrieval
4. t-SNE visualisation of the feature space
5. Cosine similarity search — finding the most similar image to a query

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

from configs.config import DATASET_CSV, FEATURES_NPY
print('Libraries loaded ✅')

## 1. Load Dataset & Build TF Image Pipeline

In [ ]:
from src.preprocessing.image_preprocessor import build_tf_dataloader, display_sample_images

df = pd.read_csv(DATASET_CSV)
print(f'Dataset: {len(df)} images')

tf_loader = build_tf_dataloader(df)
display_sample_images(tf_loader, n=4)

## 2. Load ViT Model

In [ ]:
from src.models.feature_extractor import load_vit_model

vit_model = load_vit_model()
print(f'ViT model loaded on: {next(vit_model.parameters()).device}')

## 3. Extract & Save Features

In [ ]:
from src.models.feature_extractor import extract_and_save_features

# This may take several minutes depending on your GPU/CPU
features = extract_and_save_features(tf_loader, model=vit_model)
print(f'Features shape: {features.shape}')  # (N, 197, 768)

## 4. t-SNE Visualisation of Feature Space

In [ ]:
features = np.load(FEATURES_NPY)
flat     = features.reshape(features.shape[0], -1)

# Use a subset to keep t-SNE tractable
subset = flat[:min(200, len(flat))]

tsne        = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_result = tsne.fit_transform(subset)

plt.figure(figsize=(10, 8))
sns.scatterplot(x=tsne_result[:, 0], y=tsne_result[:, 1],
                palette='viridis', legend=False, alpha=0.8)
plt.title('t-SNE of ViT Extracted Features', fontsize=14, fontweight='bold')
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.grid(True, linestyle='--', linewidth=0.5)
sns.despine()
plt.tight_layout()
plt.savefig('../results/figures/tsne_features.png', dpi=150)
plt.show()

## 5. Cosine Similarity Search (Demo)

In [ ]:
from src.models.feature_extractor import find_similar_image

# Replace with any image path from your dataset
query_path = df.iloc[0]['image_path']
print(f'Query image: {query_path}')

best_idx, similarity = find_similar_image(query_path, model=vit_model)

print(f'\nMost similar image → index {best_idx}')
print(f'Cosine similarity  → {similarity:.4f}')
print(f'\nMatched clinical data:')
print(df.iloc[best_idx][['CaseID', 'input_text', 'target_text']])